In [ ]:
import tensorflow as tf
from keras import layers

transform_pipeline = tf.keras.Sequential([
    layers.Resizing(224, 244),
    layers.Rescaling(1 / 255),
    layers.Normalization(
        mean=[0.485, 0.456, 0.406],
        variance=[0.229**2, 0.224**2, 0.225**2]
    )
])


transform_pipeline_2 = tf.keras.Sequential([
    layers.Resizing(224, 244),
    layers.Rescaling(1 / 255)
])


transform_pipeline_3 = tf.keras.Sequential([
    layers.Resizing(244, 244),
    layers.Rescaling(1 / 255),
    layers.Normalization(
        mean=[0.5, 0.5, 0.5],
        variance=[0.5**2, 0.5**2, 0.5**2]
    )
])

In [ ]:
import tensorflow as tf
from keras import layers

transform_pipeline = tf.keras.Sequential([
    layers.RandomCrop(244, 244),
    layers.RandomTranslation(
        height_factor=0.1,
        width_factor=0.1
    ),
    layers.RandomRotation(15 / 360),
    layers.RandomFlip("horizontal"),
    layers.Rescaling(1 / 255)
])

In [ ]:
import tensorflow as tf
import keras

def SimpleResNet(num_classes=10):

    inputs = keras.layers.Input(shape=(224, 224, 3))

    x = layers.Conv2D(64, 3, padding="same")(inputs)
    x = layers.ReLU()(x)

    identity1 = x

    out1 = layers.Conv2D(64, 3, padding="same")(x)
    out1 = layers.ReLU()(out1)
    out1 = layers.Conv2D(64, 3, padding="same")(out1)

    x = layers.Add()([out1, identity1])
    x = layers.ReLU()(x)

    identity2 = x

    out2 = layers.Conv2D(64, 3, padding="same")(x)
    out2 = layers.ReLU()(out2)
    out2 = layers.Conv2D(64, 3, padding="same")(out2)

    x = layers.Add()([out2, identity2])
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(num_classes)(x)

    return keras.Model(inputs, outputs)

In [ ]:
import tensorflow as tf
import keras

def SimpleBatchNormDropoutNet(num_classes=10):

    inputs = keras.Input(shape=(224, 224, 3))

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.SpatialDropout2D(0.2)(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.SpatialDropout2D(0.2)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes)(x)

    return keras.Model(inputs, outputs)

In [ ]:
import tensorflow as tf
import keras
from keras import layers

def FullResNet(num_classes=10):

    inputs = keras.Input(shape=(224, 224, 3))

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    identity1 = x

    out1 = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    out1 = layers.BatchNormalization()(out1)
    out1 = layers.ReLU()(out1)

    out1 = layers.Conv2D(64, 3, padding="same", use_bias=False)(out1)
    out1 = layers.BatchNormalization()(out1)

    x = layers.Add()([out1, identity1])
    x = layers.ReLU()(x)

    identity2 = x

    out2 = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    out2 = layers.BatchNormalization()(out2)
    out2 = layers.ReLU()(out2)

    out2 = layers.Conv2D(64, 3, padding="same", use_bias=False)(out2)
    out2 = layers.BatchNormalization()(out2)

    x = layers.Add()([out2, identity2])
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(num_classes)(x)

    return keras.Model(inputs, outputs)


if __name__ == "__main__":

    model = FullResNet(num_classes=10)

    optimizer = keras.optimizers.Adam(learning_rate=0.001)

    criterion = keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )

    scheduler = keras.optimizers.schedules.StepDecay(
        initial_learning_rate=0.001,
        decay_steps=5,
        decay_rate=0.5
    )

    optimizer.learning_rate = scheduler

    num_epochs = 10

    inputs = tf.random.normal((4, 224, 224, 3))
    labels = tf.random.uniform(
        (4,),
        minval=0,
        maxval=10,
        dtype=tf.int32
    )

    for epoch in range(num_epochs):

        with tf.GradientTape() as tape:

            outputs = model(inputs, training=True)
            loss = criterion(labels, outputs)

        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        current_lr = scheduler(epoch * 1.0)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {float(loss.numpy()):.4f} | "
            f"Active LR : {float(current_lr.numpy())}"
        )

In [ ]:
import tensorflow as tf
import keras
from keras import layers

def FullResNet(num_classes=10):

    inputs = keras.Input(shape=(224, 224, 3))

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    identity1 = x

    out1 = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    out1 = layers.BatchNormalization()(out1)
    out1 = layers.ReLU()(out1)

    out1 = layers.Conv2D(64, 3, padding="same", use_bias=False)(out1)
    out1 = layers.BatchNormalization()(out1)

    x = layers.Add()([out1, identity1])
    x = layers.ReLU()(x)

    identity2 = x

    out2 = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    out2 = layers.BatchNormalization()(out2)
    out2 = layers.ReLU()(out2)

    out2 = layers.Conv2D(64, 3, padding="same", use_bias=False)(out2)
    out2 = layers.BatchNormalization()(out2)

    x = layers.Add()([out2, identity2])
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(num_classes)(x)

    return keras.Model(inputs, outputs)


if __name__ == "__main__":

    model = FullResNet(num_classes=10)

    optimizer = keras.optimizers.Adam(
        learning_rate=0.001,
        weight_decay=1e-4
    )

    criterion = keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )

    num_epochs = 10

    for epoch in range(num_epochs):

        inputs = tf.random.normal((4, 224, 224, 3))

        labels = tf.random.uniform(
            (4,),
            minval=0,
            maxval=10,
            dtype=tf.int32
        )

        with tf.GradientTape() as tape:

            outputs = model(inputs, training=True)
            loss = criterion(labels, outputs)

        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {float(loss.numpy()):.4f}"
        )

In [ ]:
import tensorflow as tf
import keras
from keras import layers

def FullResNet(num_classes=10):

    inputs = keras.Input(shape=(224, 224, 3))

    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    identity1 = x

    out1 = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    out1 = layers.BatchNormalization()(out1)
    out1 = layers.ReLU()(out1)

    out1 = layers.Conv2D(64, 3, padding="same", use_bias=False)(out1)
    out1 = layers.BatchNormalization()(out1)

    x = layers.Add()([out1, identity1])
    x = layers.ReLU()(x)

    identity2 = x

    out2 = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    out2 = layers.BatchNormalization()(out2)
    out2 = layers.ReLU()(out2)

    out2 = layers.Conv2D(64, 3, padding="same", use_bias=False)(out2)
    out2 = layers.BatchNormalization()(out2)

    x = layers.Add()([out2, identity2])
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling2D()(x)

    outputs = layers.Dense(num_classes)(x)

    return keras.Model(inputs, outputs)


if __name__ == "__main__":

    model = FullResNet(num_classes=10)

    optimizer = keras.optimizers.Adam(learning_rate=0.001)

    criterion = keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    )

    num_epochs = 10

    for epoch in range(num_epochs):

        inputs = tf.random.normal((4, 224, 224, 3))

        labels = tf.random.uniform(
            (4,),
            minval=0,
            maxval=10,
            dtype=tf.int32
        )

        with tf.GradientTape() as tape:

            outputs = model(inputs, training=True)
            loss = criterion(labels, outputs)

        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        gradients, _ = tf.clip_by_global_norm(
            gradients,
            clip_norm=1.0
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        print(
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Loss: {float(loss.numpy()):.4f}"
        )

In [ ]:
import tensorflow as tf
import keras

class FullResNet(keras.Model):
    def __init__(self, num_classes):
        super().__init__()

        self.conv1 = layers.Conv2D(64, 3, padding="same")
        self.bn1 = layers.BatchNormalization()

        self.block1_conv1 = layers.Conv2D(64, 3, padding="same")
        self.block1_bn1 = layers.BatchNormalization()
        self.block1_conv2 = layers.Conv2D(64, 3, padding="same")
        self.block1_bn2 = layers.BatchNormalization()

        self.block2_conv1 = layers.Conv2D(64, 3, padding="same")
        self.block2_bn1 = layers.BatchNormalization()
        self.block2_conv2 = layers.Conv2D(64, 3, padding="same")
        self.block2_bn2 = layers.BatchNormalization()

        self.relu = layers.ReLU()
        self.pool = layers.GlobalAveragePooling2D()
        self.fc = layers.Dense(num_classes)

    def residual_block(self, x, conv1, bn1, conv2, bn2):
        identity = x

        x = conv1(x)
        x = bn1(x)
        x = self.relu(x)

        x = conv2(x)
        x = bn2(x)

        x = layers.Add()([x, identity])
        return self.relu(x)

    def call(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.residual_block(
            x, self.block1_conv1, self.block1_bn1,
            self.block1_conv2, self.block1_bn2
        )

        x = self.residual_block(
            x, self.block2_conv1, self.block2_bn1,
            self.block2_conv2, self.block2_bn2
        )

        x = self.pool(x)
        return self.fc(x)


model = FullResNet(num_classes=10)

optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

x = tf.random.normal((4, 224, 224, 3))
y = tf.random.uniform((4,), maxval=10, dtype=tf.int32)

with tf.GradientTape() as tape:
    predictions = model(x, training=True)
    loss = loss_fn(y, predictions)

gradients = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(gradients, model.trainable_variables))

print(f"Loss : {loss.numpy():.4f}")

In [ ]:
import tensorflow as tf
import keras

train_transform_pipeline = keras.Sequential([
    layers.Resizing(224, 224),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(15 / 360),
    layers.RandomZoom((-0.2, 0.0)),
    layers.RandomTranslation(0.1, 0.1),
    layers.Rescaling(1 / 255),
    layers.Normalization(
        mean=[0.485, 0.456, 0.406],
        variance=[0.229**2, 0.224**2, 0.225**2]
    )
])

class FullPipelineNet(keras.Model):
    def __init__(self, num_classes=10):
        super().__init__()

        self.initial_conv = layers.Conv2D(64, 3, padding="same", use_bias=False)
        self.initial_bn = layers.BatchNormalization()
        self.relu = layers.ReLU()

        self.b1_conv1 = layers.Conv2D(64, 3, padding="same", use_bias=False)
        self.b1_bn1 = layers.BatchNormalization()
        self.b1_conv2 = layers.Conv2D(64, 3, padding="same", use_bias=False)
        self.b1_bn2 = layers.BatchNormalization()

        self.b2_conv1 = layers.Conv2D(64, 3, padding="same", use_bias=False)
        self.b2_bn1 = layers.BatchNormalization()
        self.b2_conv2 = layers.Conv2D(64, 3, padding="same", use_bias=False)
        self.b2_bn2 = layers.BatchNormalization()

        self.dropout2d = layers.SpatialDropout2D(0.2)
        self.pool = layers.GlobalAveragePooling2D()
        self.dropout = layers.Dropout(0.5)
        self.fc = layers.Dense(num_classes)

    def call(self, x, training=False):

        x = self.initial_conv(x)
        x = self.initial_bn(x, training=training)
        x = self.relu(x)

        identity1 = x

        out1 = self.b1_conv1(x)
        out1 = self.b1_bn1(out1, training=training)
        out1 = self.relu(out1)
        out1 = self.b1_conv2(out1)
        out1 = self.b1_bn2(out1, training=training)

        x = self.relu(out1 + identity1)
        x = self.dropout2d(x, training=training)

        identity2 = x

        out2 = self.b2_conv1(x)
        out2 = self.b2_bn1(out2, training=training)
        out2 = self.relu(out2)
        out2 = self.b2_conv2(out2)
        out2 = self.b2_bn2(out2, training=training)

        x = self.relu(out2 + identity2)
        x = self.dropout2d(x, training=training)

        x = self.pool(x)
        x = self.dropout(x, training=training)
        x = self.fc(x)

        return x


if __name__ == "__main__":

    device = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"

    with tf.device(device):

        model = FullPipelineNet(num_classes=10)

        optimizer = keras.optimizers.Adam(
            learning_rate=0.001,
            weight_decay=1e-4,
            global_clipnorm=1.0
        )

        criterion = keras.losses.SparseCategoricalCrossentropy(
            from_logits=True
        )

        scheduler = keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=0.001,
            decay_steps=5,
            decay_rate=0.5,
            staircase=True
        )

        optimizer.learning_rate = scheduler

        num_epochs = 10

        for epoch in range(num_epochs):

            inputs = tf.random.normal((4, 224, 224, 3))
            labels = tf.random.uniform((4,), 0, 10, dtype=tf.int32)

            with tf.GradientTape() as tape:

                outputs = model(inputs, training=True)
                loss = criterion(labels, outputs)

            gradients = tape.gradient(loss, model.trainable_variables)

            optimizer.apply_gradients(
                zip(gradients, model.trainable_variables)
            )

            current_lr = optimizer.learning_rate

            print(
                f"Epoch {epoch + 1:02d} | "
                f"Loss: {loss.numpy():.4f} | "
                f"Current LR: {current_lr.numpy():.6f}"
            )